# End-to-End RAG Pipeline using Langchain

Expert Question Answerer for InsureLLM

LangChain implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace all-MiniLM-L6-v2/ollama nomic-embed-text)

In [1]:
# Importing Libraries
from langchain_ollama import ChatOllama, OllamaEmbeddings

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [2]:
OLLAMA_MODEL = "llama3.1"
MODEL = "meta-llama/Llama-3.1-8B"
db_name = "/home/alexender/Desktop/Projects/My_projects/Data/simple_rag_vector_db"
EMBED_MODEL = "nomic-embed-text"

In [3]:
# Connecting to Chroma

# embeddings = OllamaEmbeddings(model=EMBED_MODEL)
embeddings = HuggingFaceEmbeddings(model="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=db_name, embedding_function=embeddings, collection_name="Test_docs")

##### Set up the 2 key LangChain objects: retriever and llm
A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers
- Some people describe temperature as being like 'creativity' but that's not quite right

- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in upcoming weeks. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [4]:
retriever = vectorstore.as_retriever()
llm = ChatOllama(temperature=0, model=OLLAMA_MODEL)

#### These LangChain objects implement the method `invoke()`

In [5]:
retriever.invoke("Who is Avery?")

[Document(id='540d4496-1090-4adc-9787-ac773ae7b55e', metadata={'source': '/home/alexender/Desktop/Projects/My_projects/Data/knowledge-base/employees/Avery Lancaster.md', 'doc_type': 'employees'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nA

In [6]:
llm.invoke("Who is Avery?")

AIMessage(content='Avery can refer to several things, so I\'ll provide a few possible answers:\n\n1. **Given name**: Avery is a given name that originated from Old English and means "ruler of the elves" or "elf counsel." It\'s a popular name for both boys and girls.\n2. **Surname**: Avery is also a common surname of English origin, often associated with Scottish and Irish families.\n3. **Fictional characters**:\n\t* Avery Frost is a character from the TV show "Pretty Little Liars."\n\t* Avery Bradley is a fictional character in the TV series "Grey\'s Anatomy."\n4. **Real people**:\n\t* Avery Bradley is an American professional basketball player who plays for the Los Angeles Clippers.\n\t* Avery Singer is an American artist known for her abstract paintings and sculptures.\n5. **Other meanings**: In some contexts, Avery can refer to a specific location or entity, such as Avery Island (a small island in Louisiana) or Avery Dennison (an American multinational corporation).\n\nWithout more 

### Now we will put everything together as one

In [7]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [8]:
def answer_question(question: str, history):
    docs = retriever.invoke(question) # Retrieve the relevent doccs
    context = "\n\n".join(doc.page_content for doc in docs) # extracting the docs and joining it as one 
    # print(context)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context) # Putting those docs in system prompt
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)]) # Invoking the LLM for answer with context
    return response.content

In [9]:
answer_question("Who is Avery Lancaster?", [])

"Avery Lancaster is the Co-Founder and Chief Executive Officer (CEO) of Insurellm. She has been instrumental in guiding the company's growth and success since its founding in 2015. With her expertise in insurance technology and innovative leadership strategies, she has positioned Insurellm as a leading player in the industry.\n\nWould you like to know more about Avery's background or accomplishments?"

#### Creating a UI with Gradio

In [10]:
gr.ChatInterface(answer_question).launch()

/home/alexender/Desktop/Projects/My_projects/envs/Torch_env/lib/python3.12/site-packages/gradio/chat_interface.py:345: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
